In [1]:
import astropy
import pysas
import paramiko
import os
import tarfile
import subprocess
import shutil
import numpy as np
from collections import defaultdict
from astropy.io.misc import fnunpickle
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.time import Time
import matplotlib.pyplot as plt
import time
import threading
#import keyboard
import pandas as pd
import scipy.optimize as opt
from matplotlib.lines import Line2D
%matplotlib qt

In [ ]:
# Defining working path
# Defined before in the reduction script

working_path = "/work2/sruedavargas/SAS-software"

binn = 5000 #in seconds
max_pause = 2500 #in sec
tolerance_per = 20 #ask the user for percentage

In [45]:
products = working_path + '/products'
spectra_files = products + '/spectra_files'

tolerance = tolerance_per/100
tolerance_val=binn*tolerance
low_lim= binn - (tolerance_val)
up_lim= binn + (tolerance_val)

In [41]:
#####  GTI BINNING #######

In [42]:
# Copying final cleaned photon event table to a new folder within the products folder. 
# The combined gti tables will also be copied 


if os.path.exists(spectra_files):
    shutil.rmtree(spectra_files)
    os.mkdir(spectra_files)
else:
    os.mkdir(spectra_files)


for file in filtered_pet:
    shutil.copy(file, spectra_files)
    
for file in os.listdir(products):
    if ('gti_combined' in file) or (file.startswith('image') and '0.2_12keV' in file):
        shutil.copy(file, spectra_files)
        
for file in os.listdir(spectra_files):
    
    if 'gti_combined' in file:
    
        obs_id = f"{file.split('_')[0]}"
        file_path = os.path.join(spectra_files,file)
        if obs_id in prob_chip_obid:
            os.remove(file_path)
          
    if file.startswith('image') and ('0.2_12keV' in file):
        
        obs_id = f"{file.split('_')[1]}"
        file_path = os.path.join(spectra_files,file)
        if obs_id in prob_chip_obid:
            os.remove(file_path)
        
        

In [43]:
os.chdir(spectra_files)

In [44]:
combined_pn_gti=[]

for file in os.listdir(spectra_files):
    if 'PN_gti_combined' in file:
        combined_pn_gti.append(file)
        

In [46]:
def gti_gen(file):
    
    with fits.open(file) as f:
        data = f[1].data
        start_old=data.field(0)
        stop_old=data.field(1)
        #row=data[0]
        col_time_int=[]
        col_pause_int=[]
        for i in range(len(data)):
            start = data[i][0]
            stop = data[i][1]
            time_int_calc = stop - start
            col_time_int.append(time_int_calc)

            if i<(len(data)-1):
                start_next = data[i+1][0]
                pause_int_calc = start_next - stop

            elif i == (len(data)-1):
                start_next = None
                pause_int_calc = None


            col_pause_int.append(pause_int_calc)
            
    old_gti= np.column_stack((start_old, stop_old, col_time_int, col_pause_int)) 
    old_gti_t= pd.DataFrame(old_gti, columns=['start','stop','duration','pause'])

    return(old_gti_t)


In [47]:
for file in combined_pn_gti:
    
    obs_id = f"{file.split('_')[0]}"
    cam = f"{file.split('_')[1]}"
    rn = os.path.splitext(file)[0]
    thres = f"{rn.split('_')[4]}"
        
    gti_gen_rel=gti_gen(file)
    
    group_start = gti_gen_rel['start'][0]
    
####    
    
    group_stop = None
    new_gti = []
    duration_hole = []
    group=0
    grouped_new_gti=pd.DataFrame(columns=["group", "start", "stop", "duration"])


    for index, row in gti_gen_rel.iterrows():
    
        if row['pause'] is not None and row['pause']<= max_pause:
        
        
            new_row = {"group":group, "start":row['start'] , "stop":row['stop'], "duration":row['duration']}
            grouped_new_gti.loc[len(grouped_new_gti)] = new_row
  
            group_stop=None
        
        if row['pause'] is None or row['pause'] > max_pause:
        
            new_row = {"group":group, "start":row['start'] , "stop":row['stop'], "duration":row['duration']}
            grouped_new_gti.loc[len(grouped_new_gti)] = new_row
        
            group_stop=row['stop']
            new_gti.append((group_start, group_stop))
        
      
            if index + 1 < len(gti_gen_rel):
            
                group_start=gti_gen_rel.start.iloc[index+1]
                group_stop=None

                group += 1

            
            else:
            
                group_start = None
                group_stop=None    

                group = None


           
            
    new_gti_df = pd.DataFrame(new_gti, columns=["start", "stop"])


    new_durations_with_hole=[]

    for index, row in new_gti_df.iterrows():
    
        duration = row['stop']-row['start']
        new_durations_with_hole.append(duration)
    

    new_gti_df["duration_with_hole"]=new_durations_with_hole

    new_gti_df.reset_index(drop=True, inplace=True)
    pd.options.display.float_format = '{:.2f}'.format
    
    
####

    group_ex=grouped_new_gti.groupby('group')

    for name, group in group_ex:
        duration_data = group['duration'].sum()
    
        if duration_data < low_lim:
        
            for index,row in grouped_new_gti.iterrows():
            
                if row['group']==name:
                
                    grouped_new_gti.drop(index, inplace=True)
            
                
            
    grouped_new_gti.reset_index(drop=True, inplace=True)
    pd.options.display.float_format = '{:.2f}'.format
    
    
####

    group_ex1=grouped_new_gti.groupby('group')

    def_new_gti = []
    
    for name, group in group_ex1:
   
        duration_data = group['duration'].sum()
    
        def rem(x):
            y = duration_data%x
            return(y)
    
    bin_opt = opt.fminbound(rem,x1=low_lim,x2=up_lim)
    
    sum_duration=0
    gti_start = group['start'].iloc[0]
    gti_stop = None
    
    for index, row in group.iterrows():
        
        sum_duration += row['duration']
        
        if sum_duration < bin_opt:
            
            gti_stop = None
            
            
        elif sum_duration == bin_opt:
            
            gti_stop = row['stop']
            
            def_new_gti.append((gti_start, gti_stop))
            
            gti_start = group.start.iloc[index+1]
            gti_stop = None
            
            
        else:


            prev_sum_dur = sum_duration - row['duration']
            missing = bin_opt - prev_sum_dur
            gti_stop = row['start']+missing

            def_new_gti.append((gti_start, gti_stop))
            

            gti_start = gti_stop
            gti_stop = None
            sum_duration = row['stop']-gti_start
            
            if sum_duration > bin_opt:
                
                array = np.arange(gti_start,row['stop'],bin_opt)
                sub_new_gti = [(array[i], array[i+1]) for i in range(len(array)-1)]
                
                def_new_gti.extend(sub_new_gti)

                gti_start = array[len(array)-1]
                gti_stop = None
        
   
                remainder = row['stop']-array[len(array)-1]
                sum_duration = remainder
        
    def_new_gti_df = pd.DataFrame(def_new_gti, columns=["START", "STOP"])
    
    col1 = fits.Column(name='START', format='D', array=def_new_gti_df['START'])

    col2 = fits.Column(name='STOP', format='D', array=def_new_gti_df['STOP'])


    cols = fits.ColDefs([col1, col2])
    hdu = fits.BinTableHDU.from_columns(cols)


    hdu.header['TUNIT1'] = 's'
    hdu.header['TUNIT2'] = 's'
    hdu.header['EXTNAME'] = 'STDGTI'
    hdu.header['HDUCLASS'] = 'OGIP'
    hdu.header['HDUCLAS1'] = 'GTI'
    hdu.header['HDUCLAS2'] = 'STANDARD'
    hdu.header['TIMEUNIT'] = 's'


    hdu.writeto(f"bin_gti_{obs_id}_{cam}_{thres}.fits", overwrite=True)
    
##PLOTTING    

    fig, ax = plt.subplots()
    

    
# Final bins 

    for index, row in def_new_gti_df.iterrows():
      
        if index % 2 == 0:  
            y_value = 1.18
        else:  
            y_value = 1.16
        ax.hlines(y_value, row['START'], row['STOP'], color='blue')
    
# Groups (long GTIs) excluding those shorter than bin length
    
    for index, row in grouped_new_gti.iterrows():
        ax.hlines(1.2, row['start'], row['stop'], color='green')    

        
#Long GTI (groups)
    for index, row in new_gti_df.iterrows():
        ax.hlines(1.22, row['start'], row['stop'], color='orange')
    
    
#Original GTI table

    for index, row in gti_gen_rel.iterrows():
        
        global start_value
        global stop_value
        start_value = gti_gen_rel.loc[0]['start']
        stop_value = gti_gen_rel.loc[len(gti_gen_rel)-1]['stop']
        ax.hlines(1.24, row['start'], row['stop'], color='red')
    

    # plt.xlim(np.min(3.3360e+08), np.max(3.3362e+08))    
    plt.ylim(np.min(1), np.max(1.4))
    ax.set_xlabel('MJD')
    locs, labels = plt.xticks()
    plt.setp(labels, rotation=90)
    ax.set_yticklabels([])
    ax.yaxis.set_visible(False)

    legend_elements = [
    
        Line2D([0], [0], color='red', lw=1, label='original GTI'),
        Line2D([0], [0], color='orange', lw=1, label='GTI allowed to merge'),
        Line2D([0], [0], color='green', lw=1, label='GTI containig enough data'),
        Line2D([0], [0], color='blue', lw=1, label='final binned GTI')
    
    ]

    ax.legend(handles=legend_elements, loc='upper right')
    
    ax2 = ax.twiny()

    whole=int((stop_value-start_value)/1000)
    sec_ax = np.arange(0,whole, 10)
    ax2.set_xticks(sec_ax)
    ax2.set_xlabel('ks')

    plt.savefig(f"bin_gti_{obs_id}_{cam}_{thres}.png")
    
    plt.close()
   

In [48]:
#Combining binned GTI table for PN with MOS1 and MOS2

bin_pn_gti_fits=[]
combined_gti_mos1=[]
combined_gti_mos2=[]

for file in os.listdir():
    
    if file.startswith('bin_gti') and file.endswith('.fits') and 'PN' in file:
        bin_pn_gti_fits.append(file)
        
    if 'MOS1_gti_combined' in file:
        combined_gti_mos1.append(file)
        
    if 'MOS2_gti_combined' in file:
        combined_gti_mos2.append(file)
        

    
matrix_bin_gti=[]

for file1 in bin_pn_gti_fits:
    
    obs_id_pn = f"{file1.split('_')[2]}"
    en1 = os.path.splitext(file1)[0]
    en_pn = f"{en1.split('_')[4]}"
    row=[file1]
    
    for file2 in combined_gti_mos1:
        
        obs_id_mos1 = f"{file2.split('_')[0]}"
        en2 = os.path.splitext(file2)[0]
        en_mos1 = f"{en2.split('_')[4]}"
        
        if obs_id_pn==obs_id_mos1 and en_pn==en_mos1:
            row.append(file2)
            
            for file3 in combined_gti_mos2:
                
                obs_id_mos2 = f"{file3.split('_')[0]}"
                en3 = os.path.splitext(file3)[0]
                en_mos2 = f"{en3.split('_')[4]}"
        
                if obs_id_pn==obs_id_mos2 and en_pn==en_mos2:
                    row.append(file3)
                    
    if len(row)>1:
        matrix_bin_gti.append(row)
        
         

In [49]:
for element in matrix_bin_gti:
    
    obs_id = f"{element[0].split('_')[2]}"
    en1 = os.path.splitext(element[0])[0]
    energy = f"{en1.split('_')[4]}"
    
    outfile_mos1 = f"bin_gti_{obs_id}_MOS1_{energy}.fits"
    outfile_mos2 = f"bin_gti_{obs_id}_MOS2_{energy}.fits"
    
    table_pn = element[0]
    table_mos1 = element[1]
    table_mos2 = element[2]
    
    
    gtimerge_path = '/net/konraid/xray/xmmsas/xmmsas_20.0.0/bin/gtimerge'
    gtimerge_mos1 = [f"{gtimerge_path} tables='{table_pn} {table_mos1}' gtitable={outfile_mos1} mergemode=and"]
    gtimerge_mos2 = [f"{gtimerge_path} tables='{table_pn} {table_mos2}' gtitable={outfile_mos2} mergemode=and"]

    subprocess.call(gtimerge_mos1, shell=True)
    subprocess.call(gtimerge_mos2, shell=True)